# 01 · 数据读取与分析

**任务**：预测美国住房贷款申请（HMDA 数据）是否被批准，输出 `accepted = 1` 的概率；评估指标 ROC AUC。训练集 105,000 行、测试集 45,000 行，原始特征 56 列。

本 notebook 读取 `data/` 中的原始 CSV，做字段审计、探索分析和数据质量检查，并生成后续所有 notebook 共用的**固定数据接口**（`artifacts/01_data_exploration/`）：统一类型的 56 列特征、标签、ID 和 seed=42 的固定五折。这里只做确定性的格式整理：不删除申请、不猜测异常值、不使用标签构造特征。

| notebook | 内容 |
|---|---|
| **01_data_exploration** | 原始数据读取、审计、探索分析、数据质量检查、固定五折 |
| **02_feature_engineering** | 全部特征的定义、三轮特征消融、保存特征库 |
| **03_model_selection** | 基线模型（逻辑回归、XGBoost、CatBoost、AutoGluon）、线性模型的数值变换、CatBoost 调参与训练方式筛选 |
| **04_final_model** | 最终五折模型、多种子、学习曲线、全量重训与最终提交 |
| **05_error_analysis** | 残差诊断、LightGBM 与神经网络、跨实验融合、伪标签、公开榜噪声与运行间差异 |

原始数据在 `data/`；每本 notebook 的结果保存在 `artifacts/` 下与它同名的目录。

默认不重写 `artifacts/01_data_exploration/`（已存在时只做逐项核对），保证之后所有实验结果目录的数据指纹不变。

In [ ]:
# 环境、路径与通用函数（5 本 notebook 相同）
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False


def find_project_dir():
    candidates = []
    if os.environ.get('HMDA_PROJECT_DIR'):
        candidates.append(Path(os.environ['HMDA_PROJECT_DIR']))
    candidates.append(Path('/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis'))
    here = Path.cwd().resolve()
    candidates += [here, *here.parents]
    for candidate in candidates:
        if (candidate / 'data' / 'train.csv').is_file() and (candidate / 'data' / 'test.csv').is_file():
            return candidate.expanduser().resolve()
    raise FileNotFoundError('找不到项目目录（应包含 data/train.csv 与 data/test.csv）；请设置环境变量 HMDA_PROJECT_DIR')


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / 'data'                                        # 原始 CSV（只读）
ARTIFACTS = PROJECT_DIR / 'artifacts'                                  # 每本 notebook 的结果写入同名子目录
PREPARED_DIR = ARTIFACTS / '01_data_exploration'                       # 01：固定数据接口
FEATURE_DIR = ARTIFACTS / '02_feature_engineering' / 'feature_store'   # 02：特征库
SEED = 42
CATBOOST_VERSION = '1.2.10'
GPU_NAME = None
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})


def ensure_catboost():
    """只在训练时调用：安装并导入固定版本的 CatBoost。"""
    global catboost
    try:
        installed = importlib.metadata.version('catboost')
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != CATBOOST_VERSION:
        if 'catboost' in sys.modules:
            raise RuntimeError('当前运行时已导入另一版 CatBoost；请重启运行时后从头运行')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'catboost=={CATBOOST_VERSION}'])
        importlib.invalidate_caches()
    import catboost as catboost_module
    if catboost_module.__version__ != CATBOOST_VERSION:
        raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}，实际为 {catboost_module.__version__}')
    catboost = catboost_module
    return catboost_module


def ensure_gpu():
    """只在训练时调用：确认 Colab 分配了 NVIDIA GPU。"""
    global GPU_NAME
    if shutil.which('nvidia-smi') is None:
        raise RuntimeError('训练需要 GPU：Colab「更改运行时类型」选 T4 GPU 后从头运行')
    probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           text=True, capture_output=True, check=False)
    if probe.returncode != 0 or not probe.stdout.strip():
        raise RuntimeError('NVIDIA GPU 不可用：请选择 T4 GPU 后从头运行')
    GPU_NAME = probe.stdout.strip().splitlines()[0]
    return GPU_NAME


print(f'项目：{PROJECT_DIR}')
print(f'Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}')

In [ ]:
import sklearn

RANDOM_STATE = 42
N_SPLITS = 5
SCHEMA_VERSION = "1.0"
MISSING_CATEGORY = "MISSING"
REBUILD_PREPARED = False   # True：重写 artifacts/01_data_exploration（manifest 的哈希会改变，已有实验目录将无法续跑）
pd.set_option("display.max_colwidth", 75)
print(f"原始数据：{DATA_DIR}\n准备数据：{PREPARED_DIR}\nsklearn {sklearn.__version__}")

## 1. 读取原始文本并验证身份与标签

全部先读成文本，避免 `00123` 丢失前导零，也避免 pandas 把合法类别当成缺失。检查字段顺序、ID、标签和提交模板；标签在转整数**之前**检查为精确的 `0`/`1`。

In [ ]:
SOURCE_FILES = ["train.csv", "test.csv", "sample_submission.csv"]
for name in SOURCE_FILES:
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(f"缺少 {DATA_DIR / name}：原始数据应放在项目的 data/ 目录。")

def read_text_csv(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)

train_raw = read_text_csv(DATA_DIR / "train.csv")
test_raw = read_text_csv(DATA_DIR / "test.csv")
sample_raw = read_text_csv(DATA_DIR / "sample_submission.csv")
assert {"id", "accepted"}.issubset(train_raw.columns)
assert "id" in test_raw and "accepted" not in test_raw
assert sample_raw.columns.tolist() == ["id", "accepted"], "提交模板字段变化，先核查接口"
feature_columns = train_raw.columns.drop(["id", "accepted"]).tolist()
assert feature_columns == test_raw.columns.drop("id").tolist(), "训练/测试特征或顺序不一致"
assert len(train_raw) > 0 and len(test_raw) > 0
for name, frame in [("train", train_raw), ("test", test_raw), ("sample", sample_raw)]:
    assert frame.columns.is_unique, f"{name}: 重复列名"
    assert frame["id"].str.strip().ne("").all() and frame["id"].is_unique, f"{name}: ID 无效"
    assert frame["id"].eq(frame["id"].str.strip()).all(), f"{name}: ID 有首尾空格"
assert set(train_raw["id"]).isdisjoint(test_raw["id"]), "train/test ID 重叠"
assert set(sample_raw["id"]) == set(test_raw["id"]), "提交模板与测试 ID 不匹配"
assert train_raw["accepted"].isin(["0", "1"]).all(), "accepted 必须为 0 或 1"
y = train_raw["accepted"].astype("int8")
assert set(y.unique()) == {0, 1}, "二分类训练数据必须包含两个类别"
train_labels = pd.DataFrame({"id": train_raw["id"], "accepted": y})
test_ids = test_raw[["id"]].copy()
display(pd.DataFrame({
    "数据": ["train", "test"], "行数": [len(train_raw), len(test_raw)],
    "特征列": [len(feature_columns)] * 2,
}))
display(y.value_counts().sort_index().rename("数量").to_frame().assign(比例=lambda t: t["数量"] / len(y)))

## 2. 字段类型与处理边界

13 个数值字段保持原单位（贷款额、房价为美元，收入为千美元/年，LTV 为百分数）；其他 43 个字段按类别处理，包括数字类别代码、DTI 区间、年龄区间和地理代码。

| 输入情况 | 统一处理 |
|---|---|
| 数值空白 | `NaN`；数值文本转换失败时直接报错 |
| 类别空白 | 字符串 `MISSING` |
| 整数类别代码 `1.0` | 规范为 `1`，不修改区间标签 |
| `county_code` / `derived_msa-md` | 五位字符串；保留 `99999` 特殊类别 |
| 不适用、无共同申请人、豁免（`1111`）等代码 | 保留各自原代码，与空白分开 |
| 负收入、零值、极端 LTV、房龄 -1 | 保留并审计，不删除也不替换 |

In [ ]:
numeric_columns = [
    "loan_amount", "loan_to_value_ratio", "property_value", "income", "loan_term",
    "total_units", "tract_population", "tract_minority_population_percent",
    "ffiec_msa_md_median_family_income", "tract_to_msa_income_percentage",
    "tract_owner_occupied_units", "tract_one_to_four_family_homes",
    "tract_median_age_of_housing_units",
]
categorical_columns = [c for c in feature_columns if c not in numeric_columns]
assert len(feature_columns) == 56, "输入 schema 变化；先审核新增/删除字段"
assert len(numeric_columns) == 13 and len(categorical_columns) == 43
assert set(numeric_columns).issubset(feature_columns)
assert set(numeric_columns).isdisjoint(categorical_columns)

schema = {
    "schema_version": SCHEMA_VERSION,
    "target_column": "accepted", "id_column": "id",
    "feature_columns": feature_columns,
    "numeric_columns": numeric_columns,
    "categorical_columns": categorical_columns,
    "numeric_dtype": "float64", "categorical_dtype": "object",
    "missing_category": MISSING_CATEGORY,
    "csv_read_options": {"dtype": "str", "keep_default_na": False},
    "geographic_columns": ["county_code", "derived_msa-md"],
    "row_alignment": "CSV row order; train_labels.id and test_ids.id are the authoritative IDs",
    "folds": {"n_splits": N_SPLITS, "shuffle": True, "random_state": RANDOM_STATE, "fold_values": [1, 2, 3, 4, 5]},
}

## 3. 用同一个确定性函数整理 train / test

函数不接收标签，也不在全数据上计算中位数、分位数或编码器。

In [ ]:
def prepare_features(raw_features):
    assert raw_features.columns.tolist() == feature_columns, "特征或列顺序变化"
    prepared = raw_features.copy()
    for column in numeric_columns:
        values = prepared[column].str.strip().replace("", np.nan)
        prepared[column] = pd.to_numeric(values, errors="raise").astype("float64")
        assert not np.isinf(prepared[column].to_numpy()).any(), f"{column}: 存在无穷值"
        assert prepared[column].isna().equals(raw_features[column].str.strip().eq("")), f"{column}: 意外缺失"
    for column in categorical_columns:
        values = prepared[column].str.strip()
        assert not values.eq(MISSING_CATEGORY).any(), f"{column}: 原始类别与缺失标记冲突"
        values = values.str.replace(r"^(\d+)\.0+$", r"\1", regex=True)
        if column in schema["geographic_columns"]:
            nonblank = values.ne("")
            assert values[nonblank].str.fullmatch(r"\d{1,5}").all(), f"{column}: 地理代码格式异常"
            values = values.where(~nonblank, values.str.zfill(5))
        prepared[column] = values.replace("", MISSING_CATEGORY).astype(object)
    return prepared

X_train = prepare_features(train_raw[feature_columns])
X_test = prepare_features(test_raw[feature_columns])
print("格式整理完成；原始数值仍保持原单位，未进行填补、缩放或编码。")

## 4. 验证信息和行顺序没有丢失

In [ ]:
for name, raw, features in [("train", train_raw, X_train), ("test", test_raw, X_test)]:
    assert features.shape == (len(raw), len(feature_columns))
    assert features.columns.tolist() == feature_columns and features.index.equals(raw.index)
    assert "id" not in features and "accepted" not in features
    assert features[numeric_columns].dtypes.eq("float64").all()
    for column in numeric_columns:
        expected = pd.to_numeric(raw[column].str.strip().replace("", np.nan), errors="raise").astype("float64")
        pd.testing.assert_series_equal(features[column], expected)
    for column in categorical_columns:
        assert features[column].map(lambda v: isinstance(v, str) and bool(v)).all()
        assert features[column].eq(MISSING_CATEGORY).equals(raw[column].str.strip().eq(""))
assert X_train.index.equals(y.index)
pd.testing.assert_series_equal(train_labels["id"], train_raw["id"])
pd.testing.assert_series_equal(test_ids["id"], test_raw["id"])

unseen_rows = []
for column in categorical_columns:
    unseen_values = sorted(set(X_test[column]) - set(X_train[column]))
    if unseen_values:
        unseen_rows.append({"字段": column, "测试新类别数": len(unseen_values),
                            "涉及测试行数": int(X_test[column].isin(unseen_values).sum()),
                            "示例": ", ".join(unseen_values[:5])})
unseen_categories = pd.DataFrame(unseen_rows, columns=["字段", "测试新类别数", "涉及测试行数", "示例"])
display(unseen_categories)
print("类型、缺失、数值保真与行对齐检查通过。")

## 5. 数据概览

下列分位数只用于描述，不作为模型阈值。对测试集只检查输入分布与接口。

In [ ]:
missing_table = pd.DataFrame({
    "字段": feature_columns,
    "训练空白数": [int(train_raw[c].str.strip().eq("").sum()) for c in feature_columns],
    "测试空白数": [int(test_raw[c].str.strip().eq("").sum()) for c in feature_columns],
})
missing_table["训练空白比例"] = missing_table["训练空白数"] / len(X_train)
missing_table["测试空白比例"] = missing_table["测试空白数"] / len(X_test)
display(missing_table.query("训练空白数 > 0 or 测试空白数 > 0").sort_values("训练空白比例", ascending=False))
display(X_train[numeric_columns].describe(percentiles=[0.5, 0.99, 0.999]).T.round(3))
category_profile = pd.DataFrame({
    "字段": categorical_columns,
    "训练类别数": [X_train[c].nunique() for c in categorical_columns],
    "最大类别比例": [X_train[c].value_counts(normalize=True).iloc[0] for c in categorical_columns],
}).sort_values("训练类别数", ascending=False)
display(category_profile.head(10))

## 6. 探索分析：哪些信息与批准有关

整体批准率约 85%。下图按关键字段列出批准率（虚线为整体批准率），可以看到：没走自动审批系统（`aus-1 = 6`）、高 DTI、LTV 缺失或过高的申请批准率明显更低。

In [ ]:
overall_rate = y.mean()
ltv = X_train['loan_to_value_ratio']
ltv_bins = pd.cut(ltv, [-np.inf, 60, 80, 90, 95, 97, 100, np.inf])
ltv_band = ltv_bins.astype(str).where(ltv.notna(), 'missing')
ltv_order = [str(c) for c in ltv_bins.cat.categories] + ['missing']
dti_order = ['<20%', '20%-<30%', '30%-<36%', *[str(v) for v in range(36, 50)], '50%-60%', '>60%', MISSING_CATEGORY]


def rate_bar(ax, values, title, order=None, top=12):
    table = pd.DataFrame({'v': np.asarray(values), 'y': y.to_numpy()}).groupby('v')['y'].agg(['mean', 'size'])
    table = table.reindex([v for v in order if v in table.index]) if order else \
        table.sort_values('size', ascending=False).head(top)
    ax.bar(range(len(table)), table['mean'], color='#4c72b0')
    ax.axhline(overall_rate, color='black', ls='--', lw=1)
    ax.set_xticks(range(len(table)), [str(v) for v in table.index], rotation=60, ha='right', fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.set_ylabel('approval rate')


fig, axes = plt.subplots(2, 3, figsize=(16, 8.5))
rate_bar(axes[0, 0], X_train['loan_type'], 'loan_type (1=conventional, 2=FHA, 3=VA, 4=RHS)')
rate_bar(axes[0, 1], X_train['aus-1'], 'aus-1 (automated underwriting system; 6 = not used)')
rate_bar(axes[0, 2], X_train['debt_to_income_ratio'], 'debt_to_income_ratio', order=dti_order)
rate_bar(axes[1, 0], ltv_band, 'loan_to_value_ratio band', order=ltv_order)
rate_bar(axes[1, 1], X_train['applicant_credit_score_type'], 'applicant_credit_score_type')
rate_bar(axes[1, 2], X_train['occupancy_type'], 'occupancy_type (1=principal residence)')
plt.tight_layout()
plt.show()
print(f'整体批准率：{overall_rate:.3f}')

**缺失本身就是信号**：关键字段缺失的申请批准率明显不同。树模型把缺失当成单独一类（`nan_mode=Min`），所以这里不填补、不删除。

In [ ]:
missing_signal = []
for column in ['loan_to_value_ratio', 'property_value', 'income', 'loan_term']:
    mask = X_train[column].isna()
    missing_signal.append({'字段': column, '缺失行数': int(mask.sum()),
                           '缺失时批准率': y[mask].mean(), '有值时批准率': y[~mask].mean()})
for column in ['debt_to_income_ratio', 'applicant_age']:
    mask = X_train[column].eq(MISSING_CATEGORY)
    missing_signal.append({'字段': column, '缺失行数': int(mask.sum()),
                           '缺失时批准率': y[mask].mean(), '有值时批准率': y[~mask].mean()})
display(pd.DataFrame(missing_signal).round(3))

fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, (column, use_log, clip) in zip(axes, [('loan_amount', True, None), ('income', True, None),
                                                ('property_value', True, None), ('loan_to_value_ratio', False, (0, 150))]):
    for label, colour, name in [(1, '#4c72b0', 'approved'), (0, '#dd8452', 'rejected')]:
        values = X_train.loc[y.eq(label) & X_train[column].notna(), column]
        if use_log:
            values = np.log10(values[values > 0])
        if clip:
            values = values.clip(*clip)
        ax.hist(values, bins=60, density=True, alpha=0.5, color=colour, label=name)
    ax.set_title(column + (' (log10)' if use_log else ', clipped 0-150'))
axes[0].legend()
plt.tight_layout()
plt.show()

## 7. 风险记录与审计计数

负收入、零收入、大额申请和 LTV 高于 100% 都保留；重复特征行先报告，不删除。

In [ ]:
audit_rows = []
def audit(dataset, check, count, action="保留并记录"):
    audit_rows.append({"dataset": dataset, "check": check, "count": int(count), "action": action})

for name, raw, features in [("train", train_raw, X_train), ("test", test_raw, X_test)]:
    audit(name, "duplicate_ids", raw["id"].duplicated().sum(), "必须为0；已校验")
    audit(name, "duplicate_feature_rows_after_first", features.duplicated().sum())
    audit(name, "income_negative", features["income"].lt(0).sum())
    audit(name, "income_zero", features["income"].eq(0).sum())
    audit(name, "ltv_above_200_percent", features["loan_to_value_ratio"].gt(200).sum())
    audit(name, "ltv_above_1000_percent", features["loan_to_value_ratio"].gt(1000).sum(), "保留；原始来源待核实")
    audit(name, "negative_housing_age", features["tract_median_age_of_housing_units"].lt(0).sum(), "保留；特殊编码待核实")
    for column in ["loan_amount", "property_value", "loan_term"]:
        audit(name, f"{column}_nonpositive", features[column].le(0).sum(), "若非0，人工核查")
    audit(name, "minority_percent_outside_0_100", (features["tract_minority_population_percent"].lt(0) | features["tract_minority_population_percent"].gt(100)).sum(), "若非0，人工核查")
    for column in feature_columns:
        audit(name, f"blank::{column}", raw[column].str.strip().eq("").sum(), "按字段类型规范缺失")

# 哈希用于紧凑的重复组合审计，不用于删除或匹配标签。
train_feature_hash = pd.util.hash_pandas_object(X_train, index=False)
test_feature_hash = pd.util.hash_pandas_object(X_test, index=False)
audit("train_test", "shared_feature_hash_values", len(set(train_feature_hash) & set(test_feature_hash)), "保留；仅诊断，存在理论哈希碰撞可能")
for row in unseen_rows:
    audit("test", f"unseen_category_rows::{row['字段']}", row["涉及测试行数"], "保留原类别，模型处理未见类别")
audit_table = pd.DataFrame(audit_rows)
display(audit_table[~audit_table["check"].str.startswith("blank::")])

## 8. 数据质量检查

- **对抗验证**：训练一个模型区分训练行和测试行。AUC ≈ 0.5 说明两者分布一致，交叉验证的结果可以代表测试集。
- **id 结构**：id 能否预测标签、相邻 id 的标签是否相关（检查按机构或时间排序造成的泄漏）。
- **派生字段一致性**：派生字段与原字段是否一致（判断数据是真实记录还是生成数据）。
- **「异常值」是不是错误**：极端或可疑取值所在组的批准率。

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score

combined = pd.concat([X_train, X_test], ignore_index=True)
encoded = combined.copy()
for column in categorical_columns:
    encoded[column] = encoded[column].astype('category').cat.codes
is_test = np.r_[np.zeros(len(X_train)), np.ones(len(X_test))]
adversarial_auc = cross_val_score(
    HistGradientBoostingClassifier(max_iter=200, random_state=RANDOM_STATE), encoded, is_test,
    cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE), scoring='roc_auc').mean()

train_id_values = pd.to_numeric(train_labels['id'])
test_id_values = pd.to_numeric(test_ids['id'])
sorted_labels = y.to_numpy()[np.argsort(train_id_values.to_numpy())].astype(float)
window = 5
neighbour_sum = pd.Series(sorted_labels).rolling(2 * window + 1, center=True, min_periods=1).sum() - sorted_labels
neighbour_count = pd.Series(np.ones(len(sorted_labels))).rolling(2 * window + 1, center=True, min_periods=1).sum() - 1
neighbour_corr = np.corrcoef(sorted_labels, neighbour_sum / neighbour_count)[0, 1]

loan_type_name = X_train['loan_type'].map({'1': 'Conventional', '2': 'FHA', '3': 'VA', '4': 'FSA/RHS'})
lien_name = X_train['lien_status'].map({'1': 'First Lien', '2': 'Subordinate Lien'})
construction_name = X_train['construction_method'].map({'1': 'Site-Built', '2': 'Manufactured'})
checks = pd.DataFrame([
    ('训练/测试对抗验证 AUC（0.5 = 分布一致）', f'{adversarial_auc:.4f}'),
    ('只用 id 预测标签的 AUC', f'{roc_auc_score(y, train_id_values):.4f}'),
    ('相邻 10 个 id 的平均标签与自身标签的相关', f'{neighbour_corr:.4f}'),
    ('id 范围：训练 / 测试', f'{train_id_values.min()}–{train_id_values.max()} / '
                           f'{test_id_values.min()}–{test_id_values.max()}'),
    ('产品类型 = 贷款类型 + 留置顺序', f"{(loan_type_name + ':' + lien_name).eq(X_train['derived_loan_product_type']).mean():.2%}"),
    ('住宅类别与施工方式一致',
     f"{X_train['derived_dwelling_category'].str.split(':').str[1].eq(construction_name).mean():.2%}"),
], columns=['检查', '结果'])
display(checks)

loan_to_value_raw = X_train['loan_amount'] / X_train['property_value']
anomalies = {
    '全部': pd.Series(True, index=X_train.index),
    'LTV 缺失': ltv.isna(),
    'LTV > 150': ltv > 150,
    '普查区人口 = 0': X_train['tract_population'].eq(0),
    '地区收入 = 0': X_train['ffiec_msa_md_median_family_income'].eq(0),
    '贷款额 / 房价 > 2': loan_to_value_raw > 2,
    '收入 ≤ 0': X_train['income'] <= 0,
}
display(pd.DataFrame([{'情况': name, '行数': int(mask.sum()), '批准率': y[mask].mean()}
                      for name, mask in anomalies.items()]).round(3))

**结论**：训练和测试分布一致、id 不含信息、派生字段与原字段一致（真实 HMDA 记录）。「异常值」所在组的批准率远低于整体，它们是**拒绝信号**而不是录入错误，因此全部保留、交给模型学习。

## 9. 固定交叉验证折

`StratifiedKFold(5, shuffle=True, random_state=42)`，折号 1–5。之后所有实验都使用这同一套折，比较时只看同一折上的配对差。

In [ ]:
assert y.value_counts().min() >= N_SPLITS
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_assignment = np.zeros(len(y), dtype="int8")
validation_seen = np.zeros(len(y), dtype="int8")
for fold, (fit_idx, valid_idx) in enumerate(cv.split(X_train, y), start=1):
    assert len(np.intersect1d(fit_idx, valid_idx)) == 0
    assert len(fit_idx) + len(valid_idx) == len(y)
    fold_assignment[valid_idx] = fold
    validation_seen[valid_idx] += 1
assert np.all(validation_seen == 1)
assert set(fold_assignment.tolist()) == set(range(1, N_SPLITS + 1))
folds = pd.DataFrame({"id": train_labels["id"], "fold": fold_assignment})
fold_profile = train_labels.assign(fold=fold_assignment).groupby("fold")["accepted"].agg(["size", "mean"])
fold_profile.columns = ["验证行数", "验证获批率"]
display(fold_profile)

## 10. 输出固定数据接口

`train_features.csv` / `test_features.csv` 只含 56 个特征；`train_labels.csv`、`test_ids.csv`、`folds.csv` 记录 ID、标签和折号；`schema.json` 描述类型，`manifest.json` 记录输入/输出的 SHA-256。后续 notebook 都会先核对这些哈希。已存在时不重写（见 `REBUILD_PREPARED`）。

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


if REBUILD_PREPARED or not (PREPARED_DIR / "manifest.json").is_file():
    PREPARED_DIR.mkdir(parents=True, exist_ok=True)
    output_frames = {
        "train_features.csv": X_train, "test_features.csv": X_test,
        "train_labels.csv": train_labels, "test_ids.csv": test_ids,
        "folds.csv": folds, "audit.csv": audit_table,
    }
    for name, frame in output_frames.items():
        destination = PREPARED_DIR / name
        temporary = destination.with_suffix(destination.suffix + ".tmp")
        frame.to_csv(temporary, index=False, na_rep="")
        temporary.replace(destination)
    (PREPARED_DIR / "schema.json").write_text(json.dumps(schema, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    manifest = {
        "schema_version": SCHEMA_VERSION,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_dir": str(DATA_DIR),
        "source_files": {name: {"sha256": sha256_file(DATA_DIR / name), "bytes": (DATA_DIR / name).stat().st_size} for name in SOURCE_FILES},
        "output_files": {name: {"sha256": sha256_file(PREPARED_DIR / name), "bytes": (PREPARED_DIR / name).stat().st_size} for name in [*output_frames, "schema.json"]},
        "n_rows": {"train": len(X_train), "test": len(X_test)},
        "n_features": len(feature_columns),
        "seed": RANDOM_STATE, "n_splits": N_SPLITS,
        "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__},
        "preprocessing": "deterministic text/type normalization only; no fitted estimator or population-derived threshold",
    }
    # manifest 最后写入；若中途失败，旧 manifest 与文件哈希不匹配，模型入口会停止。
    manifest_temp = PREPARED_DIR / "manifest.json.tmp"
    manifest_temp.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    manifest_temp.replace(PREPARED_DIR / "manifest.json")
    print(f"已保存 {len(output_frames) + 2} 个准备文件。")
else:
    manifest = json.loads((PREPARED_DIR / "manifest.json").read_text(encoding="utf-8"))
    print("artifacts/01_data_exploration 已存在，未重写；下一格用本次重新整理的结果逐项核对已保存的文件。")

## 11. 回读验证

用本次重新整理的结果，逐项核对已保存的文件：数值、类别、列顺序、ID、折号和 SHA-256。

In [ ]:
def read_prepared_features(path, saved_schema):
    frame = read_text_csv(path)
    assert frame.columns.tolist() == saved_schema["feature_columns"]
    for column in saved_schema["numeric_columns"]:
        frame[column] = pd.to_numeric(frame[column].replace("", np.nan), errors="raise").astype("float64")
    for column in saved_schema["categorical_columns"]:
        assert frame[column].ne("").all(), f"{column}: 准备文件出现空白类别"
        frame[column] = frame[column].astype(object)
    return frame

saved_schema = json.loads((PREPARED_DIR / "schema.json").read_text(encoding="utf-8"))
for name, expected in [("train_features.csv", X_train), ("test_features.csv", X_test)]:
    restored = read_prepared_features(PREPARED_DIR / name, saved_schema)
    pd.testing.assert_frame_equal(restored, expected, check_exact=False, rtol=1e-14, atol=0)
pd.testing.assert_frame_equal(read_text_csv(PREPARED_DIR / "train_labels.csv"), train_labels.astype(str))
pd.testing.assert_frame_equal(read_text_csv(PREPARED_DIR / "test_ids.csv"), test_ids)
pd.testing.assert_frame_equal(read_text_csv(PREPARED_DIR / "folds.csv"), folds.astype(str))
for name, entry in manifest["output_files"].items():
    assert sha256_file(PREPARED_DIR / name) == entry["sha256"], f"输出哈希变化：{name}"
print("准备文件回读、类型恢复、行对齐与 SHA-256 检查全部通过。")